###Job posting data was collected through web scraping from the job search section of the Saramin website, focusing on positions located in Seoul and categorized under Data Analysis, Data Engineering, Deep Learning, and Machine Learning.

###사람인 웹사이트의 채용 검색 페이지에서 웹 스크래핑을 통해 채용 공고 데이터를 수집했으며, 서울 지역의 Data Analysis, Data Engineering, Deep Learning 및 Machine Learning 직무를 대상으로 데이터를 수집했다.

In [3]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from urllib.request import urlopen

In [13]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

url = "https://www.saramin.co.kr/zf_user/jobs/list/domestic"
params = {
    "loc_mcd": "101000",
    "cat_kewd": "82,83,2248,109,108",
    "search_done": "y",
    "panel_count": "y",
}

company_list = []
title_list = []
sector_list = []
location_list = []
career_list = []
education_list = []

for page in range(1, 21):
    params["page"] = page
    response = requests.get(url, params=params, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")

    jobs = soup.find_all("div", class_="list_item")

    for job in jobs:
        company = job.find("a", class_="str_tit")
        company_list.append(company.text.strip() if company else "")

        title = job.find("div", class_="job_tit")
        title_list.append(title.find("a")["title"] if title and title.find("a") else "")

        sector = job.find("span", class_="job_sector")   #
        sector_list.append(sector.text.strip() if sector else "")

        location = job.find("p", class_="work_place")
        location_list.append(location.text.strip() if location else "")

        career = job.find("p", class_="career")
        career_list.append(career.text.strip() if career else "")

        education = job.find("p", class_="education")
        education_list.append(education.text.strip() if education else "")

    print(f"{page} 페이지 완료, 누적 {len(company_list)}개")
    time.sleep(1)

df = pd.DataFrame({
    "company": company_list,
    "title": title_list,
    "sector": sector_list,       #
    "location": location_list,
    "career": career_list,
    "education": education_list
})

print(df.columns.tolist())
print(df.shape)

1 페이지 완료, 누적 50개
2 페이지 완료, 누적 100개
3 페이지 완료, 누적 150개
4 페이지 완료, 누적 200개
5 페이지 완료, 누적 250개
6 페이지 완료, 누적 300개
7 페이지 완료, 누적 350개
8 페이지 완료, 누적 400개
9 페이지 완료, 누적 450개
10 페이지 완료, 누적 500개
11 페이지 완료, 누적 550개
12 페이지 완료, 누적 600개
13 페이지 완료, 누적 650개
14 페이지 완료, 누적 700개
15 페이지 완료, 누적 750개
16 페이지 완료, 누적 800개
17 페이지 완료, 누적 850개
18 페이지 완료, 누적 900개
19 페이지 완료, 누적 950개
20 페이지 완료, 누적 957개
['company', 'title', 'sector', 'location', 'career', 'education']
(957, 6)


In [14]:
def categorize(sector_text):
    if "데이터분석가" in sector_text:
        return "Data Analysis"
    elif "데이터엔지니어" in sector_text:
        return "Data Engineer"
    elif "딥러닝" in sector_text:
        return "Deep Learning"
    elif "머신러닝" in sector_text:
        return "Machine Learning"
    else:
        return "Other"

df["category"] = df["sector"].apply(categorize)
print(df["category"].value_counts())

category
Data Analysis       412
Data Engineer       279
Deep Learning       158
Machine Learning     62
Other                46
Name: count, dtype: int64


In [15]:
df.isnull().sum()

,0
company,0
title,0
sector,0
location,0
career,0
education,0
category,0


.describe() calculates the top/freq values independently for each column — it does not show relationships between columns. The top value in title (e.g., "Data Engineer") and the top value in category (e.g., "Data Analysis") are unrelated statistics; each simply reflects the most frequent value within that single column, not a matching pair across rows.

.describe()는 top/freq 값을 각 열마다 독립적으로 계산합니다 — 열 간의 관계를 보여주지 않습니다. title 열의 top 값(예: "Data Engineer")과 category 열의 top 값(예: "Data Analysis")은 서로 연관된 통계가 아니며, 각각 해당 열 안에서만 가장 많이 등장한 값을 나타낼 뿐, 같은 행끼리 짝지어진 정보가 아닙니다.

In [16]:
df.describe()

,company,title,sector,location,career,education,category
count,957,957,957,957,957,957,957
unique,610,950,684,50,151,6,5
top,(주)터닝포인트에이치알,Data Engineer,데이터엔지니어,서울 강남구,신입 · 경력 · 정규직,대학교(4년)↑,Data Analysis
freq,20,3,17,285,99,449,412


In [17]:
df.duplicated().sum()

np.int64(0)

In [19]:
df.to_csv("saramin_jobs.csv", index=False, encoding="utf-8-sig")